# core

> Fill in a module description here

In [ ]:
#| default_exp core

In [ ]:
#| export
from collections import namedtuple
from fastcore.utils import *

import httpx, os, time

`SolveItClient` is our main entry point. It wraps an `httpx.Client` with authentication and JSON handling. The `__call__` method handles both JSON and plain text responses automatically.

In [ ]:
#| export
class SolveItClient:
    "SolveIt API client"
    def __init__(self, url=None, token=None, timeout=30):
        url = url or os.environ.get('SOLVEIT_URL') or 'http://localhost:5001'
        self.url, self.token = url.rstrip('/') + '/', token or os.environ.get('SOLVEIT_TOKEN', 'dummy')
        self.cli = httpx.Client(base_url=self.url, timeout=timeout,
                                cookies={'_solveit': self.token},
                                headers={'Accept': 'application/json'})
    
    def __call__(self, path, **data):
        res = self.cli.post(path, data=data)
        try: return res.json()
        except: return res.text
    
    def __repr__(self): return f'SolveItClient({self.url=})'

In [ ]:
#| export
def _resp_err(res):
    "Raise an exception if `res` contains an error, otherwise return `res`."
    if isinstance(res, dict) and 'error' in res:
        err = res['error']
        if isinstance(err, str) and err.startswith('Failed to access ') and '. Does it exist?' in err:
            dlg_name = err[len('Failed to access '):].split(' in ', 1)[0]
            raise Exception(f'Dialog not found: {dlg_name}')
        raise Exception(err)
    return res

In [ ]:
prev_url = os.environ.pop('SOLVEIT_URL', None)
try:
    assert SolveItClient(token='tok').url == 'http://localhost:5001/'
finally:
    if prev_url is not None: os.environ['SOLVEIT_URL'] = prev_url

sic = SolveItClient(); sic

SolveItClient(self.url='http://localhost:6001/')

We can test out our client using the `/test_route`. If you get an error saying `No access. Please login and then retry.`, make sure your SOLVEIT_TOKEN is correct and exported/added as a secret

In [ ]:
sic('/test_route')

'here'

`Dialog` wraps a dialog name and client, caching metadata from `/curr_dialog_`. The `__getattr__`/`__getitem__`/`__dir__` pattern makes dialog fields accessible as attributes with autocomplete support.

In [ ]:
#| export
class Dialog:
    "Dialog operations"
    def __init__(self, name, cli):
        self.name, self.cli = name, cli
        self._refresh()
    
    def _refresh(self):
        self.data = self.cli('/curr_dialog_', dlg_name=self.name)
    
    def __getattr__(self, k): return self.data.get(k)
    def __getitem__(self, k): return self.data[k]
    def __dir__(self): return list(self.data.keys()) + dir(type(self))
    def __repr__(self): return f"Dialog('{self.name}', mode={self.mode})"
    @property
    def link(self):
        "Link to the dialog."
        return f'{self.cli.url}dialog_?name={self.name}'
    def _repr_markdown_(self): 
        return f"**Dialog:** <a href='{self.link}' target='_blank'>`{self.name}`</a> | **Mode:** {self.mode}"

In [ ]:
#| export
@patch
def create_dialog(self:SolveItClient, name):
    "Create a new dialog with `name` and return a `Dialog` instance."
    _resp_err(self('/create_dialog_', name=name, api=True))
    return Dialog(name, self)

In [ ]:
dlg = sic.create_dialog('folder/test'); dlg

**Dialog:** <a href='http://localhost:6001/dialog_?name=folder/test' target='_blank'>`folder/test`</a> | **Mode:** learning

`Message` represents a single message in a dialog. Like `Dialog`, it uses `__getattr__` for attribute access to message fields (content, output, msg_type, etc.). The `_refresh()` method fetches current state from the server.

In [ ]:
#| export
class Message:
    "Message operations"
    def __init__(self, id, dlg, data=None):
        self.id, self.dlg = id, dlg
        self.data = data if data else self._refresh()
    
    def _refresh(self):
        self.data = self.dlg.cli('/read_msg_', dlg_name=self.dlg.name, id_=self.id, n=0, relative=True)
        return self.data
    
    def __getattr__(self, k): return self.data.get(k)
    def __getitem__(self, k): return self.data[k]
    def __dir__(self): return list(self.data.keys()) + dir(type(self))
    def __repr__(self): return f"Message('{self.id}', type={self.msg_type})"
    @property
    def link(self):
        "Link to the message."
        return self.dlg.link + f'#{self.id}'
    def _repr_markdown_(self): 
        preview = (self.content[:50] + '...') if len(self.content or '') > 50 else self.content
        out = (self.output[:50] + '...') if len(self.output or '') > 50 else (self.output or '')
        return f"**Message:** <a href='{self.link}' target='_blank'>`{self.id}`</a> | **Type:** {self.msg_type} | `{preview}` | **Output:** `{out}`" 


In [ ]:
#| export
@patch
def add_msg(self:Dialog, content, msg_type='code', placement='at_end', heading_collapsed=0, i_collapsed=0, o_collapsed=0, id=None):
    "Add a new message to the dialog and return it."
    res = self.cli( '/add_relative_', dlg_name=self.name, content=content, msg_type=msg_type, placement=placement,
                    heading_collapsed=heading_collapsed, i_collapsed=i_collapsed, o_collapsed=o_collapsed, id_=id)
    res = _resp_err(res)
    return Message(res['id'], self)

In [ ]:
msg = dlg.add_msg('1+1'); msg

**Message:** <a href='http://localhost:6001/dialog_?name=folder/test#_5c83f3ad' target='_blank'>`_5c83f3ad`</a> | **Type:** code | `1+1` | **Output:** ``

`Messages` is a list subclass that renders as an HTML table in notebooks. It wraps raw message dicts into `Message` objects for convenient access.

In [ ]:
#| export
class Messages(list):
    def __init__(self, msgs, dlg):
        super().__init__(Message(m['id'], dlg, m) for m in msgs)
    def _repr_html_(self):
        rows = ''.join(
            f"<tr><td><a href='{m.link}' target='_blank'><code>{m.id}</code></a></td>"
            f"<td>{m.msg_type}</td><td>{(m.content or '')[:40]}</td><td>{(m.output or '')[:40]}</td></tr>" for m in self)
        return ('<div style="overflow-x:auto"><table style="border-collapse:collapse;width:100%">'
                '<tr><th>ID</th><th>Type</th><th>Content</th><th>Output</th></tr>'
                f'{rows}</table></div>').replace('<td>', '<td style="padding:6px 14px">').replace('<th>', '<th style="padding:6px 14px;text-align:left">')

In [ ]:
#| export
@patch
def find_msgs(self:Dialog, re_pattern=None, **kwargs):
    "Find messages matching `re_pattern` and return as `Messages`."
    res = _resp_err(self.cli('/find_msgs_', dlg_name=self.name, re_pattern=re_pattern, **kwargs))
    return Messages(res['msgs'], self)

In [ ]:
#| export
@patch(as_prop=True)
def messages(self:Dialog):
    "All messages in this dialog."
    return self.find_msgs()

In [ ]:
msgs = dlg.messages; msgs

ID,Type,Content,Output
_7e22d2f4,prompt,"Hi, how are you?","Hey! I'm doing great, thanks for asking"
_e3aca5ce,code,"print(""I got ran!"")",I got ran!
_5c83f3ad,code,1+1,


In [ ]:
#| export
@patch
def to_xml(self:Dialog,msg_type:str=None, # optional limit by message type ('code', 'note', or 'prompt')
    nums:bool=False, # Whether to show line numbers
    include_output:bool=False, # Include output in returned dict?
    trunc_out:bool=True, # Middle-out truncate code output to 100 characters (only applies if `include_output`)?
    trunc_in:bool=False, # Middle-out truncate cell content to 80 characters?
):
    "Return dialog messages as an XML string."
    return self.cli('/find_msgs_', dlg_name=self.name, as_xml=True, nums=nums, include_meta=False,
                    msg_type=msg_type, include_output=include_output, trunc_out=trunc_out, trunc_in=trunc_in)

In [ ]:
print(dlg.to_xml())

<msgs><prompt id="_7e22d2f4"><source>Hi, how are you?<out>Hey! I'm doing great, thanks for asking 😊 How about you? What are you working on today?

<details class='token-usage-details'><summary>$0.0147</summary>

`total=15,946 | in=15,918 | out=28 | cached=98.4% | cache_new=5 | $0.0147`

</details></out></prompt><code id="_e3aca5ce">print("I got ran!")</code><code id="_5c83f3ad">1+1</code></msgs>


In [ ]:
#| export
@patch
def toggle_header(self:Dialog, re_pattern):
    "Toggle the header collapse state of the first message matching `re_pattern`."
    msgs = self.find_msgs(re_pattern=re_pattern)
    if msgs: self.cli('/toggle_header_collapse_', dlg_name=self.name, id_=msgs[0].id)

## Message Operations

Messages support CRUD operations via the REST API.

In [ ]:
msg = msgs[0]; msg

**Message:** <a href='http://localhost:6001/dialog_?name=folder/test#_7e22d2f4' target='_blank'>`_7e22d2f4`</a> | **Type:** prompt | `Hi, how are you?` | **Output:** `Hey! I'm doing great, thanks for asking 😊 How abou...`

In [ ]:
#| export
@patch
def exec(self:Message, timeout=30, poll_interval=0.2):
    "Execute this message and poll until completion or timeout."
    self.dlg.cli('/add_runq_', dlg_name=self.dlg.name, id_=self.id, api='true')
    for _ in range(int(timeout / poll_interval)):
        time.sleep(poll_interval)
        self._refresh()
        if not self.data.get('run'): break
    return self

In [ ]:
msg.exec(); msg

**Message:** <a href='http://localhost:6001/dialog_?name=folder/test#_7e22d2f4' target='_blank'>`_7e22d2f4`</a> | **Type:** prompt | `Hi, how are you?` | **Output:** `<output result="pending" reason="incomplete"/>`

In [ ]:
ai_msg = dlg.add_msg('Hi, how are you?', msg_type='prompt'); ai_msg

**Message:** <a href='http://localhost:6001/dialog_?name=folder/test#_8e20efac' target='_blank'>`_8e20efac`</a> | **Type:** prompt | `Hi, how are you?` | **Output:** `<output result="pending" reason="incomplete"/>`

In [ ]:
ai_msg.exec(); ai_msg

**Message:** <a href='http://localhost:6001/dialog_?name=folder/test#_8e20efac' target='_blank'>`_8e20efac`</a> | **Type:** prompt | `Hi, how are you?` | **Output:** `<output result="pending" reason="incomplete"/>`

In [ ]:
#| export
MsgDiff = namedtuple('MsgDiff', ['msg', 'diff'])

@patch
def update(self:Message, **kwargs):
    "Update message fields and return a `MsgDiff` with the change."
    if 'msg_type' in kwargs: kwargs.setdefault('output', '')
    res = self.dlg.cli('/update_msg_', dlg_name=self.dlg.name, id_=self.id, log_changed=True, **kwargs)
    _resp_err(res)
    self._refresh()
    return MsgDiff(self, res.get('diff'))

In [ ]:
msg.update(content='2+2')

MsgDiff(msg=Message('_7e22d2f4', type=prompt), diff='@@ -1 +1 @@\n-Hi, how are you?\n+2+2')

In [ ]:
msg.exec(); msg

**Message:** <a href='http://localhost:6001/dialog_?name=folder/test#_7e22d2f4' target='_blank'>`_7e22d2f4`</a> | **Type:** prompt | `2+2` | **Output:** `<output result="pending" reason="incomplete"/>`

In [ ]:
#| export
@patch
def delete(self:Message):
    "Delete this message from its dialog."
    self.dlg.cli('/rm_msg_', dlg_name=self.dlg.name, msid=self.id, api='true')
    return self

In [ ]:
msg.delete()

**Message:** <a href='http://localhost:6001/dialog_?name=folder/test#_7e22d2f4' target='_blank'>`_7e22d2f4`</a> | **Type:** prompt | `2+2` | **Output:** `<output result="pending" reason="incomplete"/>`

In [ ]:
dlg.messages

## Dialog Helper Equivalents

In [ ]:
#| export
@patch(as_prop=True)
def num_content(self:Message):
    "Return content with line numbers."
    self._refresh()
    return '\n'.join(f'{i+1:4d} | {l}' for i,l in enumerate(self.content.splitlines()))

@patch
def str_replace(self:Message, old_str, new_str):
    "Replace `old_str` with `new_str` in message content (must match exactly once)."
    count = self.content.count(old_str)
    if count == 0: raise ValueError(f"Text not found: {repr(old_str)}")
    if count > 1: raise ValueError(f"Multiple matches ({count}): {repr(old_str)}")
    return self.update(content=self.content.replace(old_str, new_str, 1))

@patch
def insert_line(self:Message, insert_line, new_str):
    "Insert `new_str` at line number `insert_line`."
    lines = self.content.splitlines()
    if not (0 <= insert_line <= len(lines)): raise ValueError(f'Invalid line {insert_line}. Valid: 0-{len(lines)}')
    lines.insert(insert_line, new_str)
    return self.update(content='\n'.join(lines))

In [ ]:
#| export
def _norm_lines(n:int, start:int, end:int=None):
    "Normalize and validate line range. Returns (start, end) or raises ValueError."
    if end is None: end = start
    if end < 0: end = n + end + 1
    if not (1 <= start <= n): raise ValueError(f'Invalid start line {start}. Valid range: 1-{n}')
    if not (start <= end <= n): raise ValueError(f'Invalid end line {end}. Valid range: {start}-{n}')
    return start, end

@patch
def strs_replace(self:Message, old_strs:list, new_strs:list):
    "Replace each string in `old_strs` with corresponding string in `new_strs`."
    if len(old_strs) != len(new_strs): raise ValueError(f"Length mismatch: {len(old_strs)} vs {len(new_strs)}")
    text = self.content
    for idx,(old,new) in enumerate(zip(old_strs, new_strs)):
        count = text.count(old)
        if count == 0: raise ValueError(f"Text not found at index {idx}: {repr(old)}")
        if count > 1: raise ValueError(f"Multiple matches ({count}) at index {idx}: {repr(old)}")
        text = text.replace(old, new, 1)
    return self.update(content=text)

@patch
def replace_lines(self:Message, start_line:int, end_line:int=None, new_content:str=''):
    "Replace lines from `start_line` to `end_line` with `new_content`."
    lines = self.content.splitlines(keepends=True)
    s,e = _norm_lines(len(lines), start_line, end_line)
    if lines and new_content and not new_content.endswith('\n'): new_content += '\n'
    lines[s-1:e] = [new_content] if new_content else []
    return self.update(content=''.join(lines))

@patch
def del_lines(self:Message, start_line:int, end_line:int=None):
    "Delete lines from `start_line` to `end_line`."
    lines = self.content.splitlines(keepends=True)
    s,e = _norm_lines(len(lines), start_line, end_line)
    del lines[s-1:e]
    return self.update(content=''.join(lines))

In [ ]:
#| export
@patch
def read_msg(self:Dialog, n=0, id=None):
    "Read a single message by index `n` or `id` and return a `Message`."
    data = _resp_err(self.cli('/read_msg_', dlg_name=self.name, id_=id, n=n, relative=True))
    return Message(data['id'], self, data)

In [ ]:
#| export
@patch
def stop(self:Dialog):
    "Stop the dialog kernel."
    self.cli('/stop_', dlg_name=self.name); return self

@patch
def reset(self:Dialog):
    "Reset the dialog instance."
    self.cli('/reset_', dlg_name=self.name); return self

@patch
def run_all(self:Dialog):
    "Run all code messages in the dialog from top to bottom."
    self.cli('/run_all_', dlg_name=self.name); return self

In [ ]:
dlg.add_msg('print("I got ran!")')

**Message:** <a href='http://localhost:6001/dialog_?name=folder/test#_6072d578' target='_blank'>`_6072d578`</a> | **Type:** code | `print("I got ran!")` | **Output:** ``

In [ ]:
dlg.run_all(); dlg.messages

ID,Type,Content,Output
_800d22d1,prompt,"Hi, how are you?","Hey! I'm doing great, thanks for asking"
_6072d578,code,"print(""I got ran!"")",I got ran!


In [ ]:
#| export
@patch
def delete(self:Dialog):
    "Delete this dialog."
    return self.cli('/rm_dialog_', name=self.name, api='true')

In [ ]:
dlg.delete()

{'success': 'deleted "/home/natedawg/folder/test"'}

# fin

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()